In [ ]:
import os
import torch
import torchvision
import yaml
import numpy as np
import logging
import matplotlib.pyplot as plt

from time import time
from omegaconf import OmegaConf
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image
from torchvision import transforms
from torchvision.utils import make_grid, save_image
from diffusers import DDIMScheduler, DDIMPipeline

from evaluation import compute_validity, calculate_fid, compute_kid
from data import get_dataset
from guidance import SpectralGuidance

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

In [ ]:
celebahq_dir = "./runs/celeba-hq-mask/phi_1000_1_k512_chunks16"
cifar10_dir = "./runs/cifar10/phi_1000-1_k512_chunks2"
experiment_dir = cifar10_dir

cfg = OmegaConf.load(os.path.join(experiment_dir, ".hydra/config.yaml"))

# Load data and spectral guidance pipeline

In [ ]:
spectral_guidance = SpectralGuidance(
    model_path=experiment_dir,
    guidance_tmin=0,
    guidance_tmax=1000,
    device=device,
)
spectral_guidance.set_timesteps(100)

dataset = get_dataset(
    dataset=spectral_guidance._cfg.dataset.name,
    split="train",
    augment=False,
    cache_dir=spectral_guidance._cfg.dataset.cache_dir,
)

# Spectral Guidance DDIM

In [ ]:
print([(i,n) for i,n in enumerate(dataset.classes)])

In [ ]:
target_idxs = ["+9"]
p_y_x0 = dataset.get_guidance(int(target_idxs[0][1:])) if target_idxs[0][0] == "+" else 1 - dataset.get_guidance(int(target_idxs[0][1:]))
for target_idx in target_idxs[1:]:
    p_y_x0 *= dataset.get_guidance(int(target_idx[1:])) if target_idx[0] == "+" else 1 - dataset.get_guidance(int(target_idx[1:]))
p_y_x0 = p_y_x0.to(device)
print(p_y_x0.sum().item())

In [ ]:
spectral_guidance.manual_seed(0)
spectral_guidance.set_guidance(p_y_x0)

samples = spectral_guidance.sample(
    num_samples=20,
    guidance_strength=10,
    eta=1.0,
    batch_size=128,
    return_posterior_mean=False,
    guidance_tmin=0,
    guidance_tmax=1000,
)
gen_images = spectral_guidance.to_pil_image(samples)

imgs = torch.stack([transforms.ToTensor()(img) for img in gen_images])
grid = make_grid(imgs[:100], nrow=6, padding=0)

plt.figure(figsize=(12,12))
plt.imshow(grid.permute(1,2,0))
plt.axis("off")